# Artha Actions E2E: Initiate → Fund → Deliver → Release

Tiny, runnable notebook that hits a local **actions-server** to simulate a full escrow flow.

## What this notebook does
1. Health check
2. Initiate a deal (server returns a `dealId` and possibly a Blink/tx)
3. Build a funding transaction (if you didn’t use Blink from step 2)
4. Mark delivered (attach evidence CID)
5. Release funds to the seller
6. Fetch final status

### Requirements
- A running actions-server pointing at a devnet/testnet RPC
- Python 3.10+
- `requests`, `python-dotenv` (optional)

### Safety tips
- Use tiny amounts on devnet
- Never paste real secrets into notebooks
- Treat API responses as untrusted; this notebook prints compact views only

In [ ]:
# If needed, install deps right from the notebook (comment out if you prefer your own env)
import sys, subprocess
def _pip(pkg: str):
    try:
        __import__(pkg.split("==")[0].split("[")[0])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

_pip("requests")
try:
    import dotenv  # optional
except Exception:
    _pip("python-dotenv")

In [ ]:
import os, json, time, typing as t
from dataclasses import dataclass
import requests
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

def env(name: str, default: str = "") -> str:
    val = os.getenv(name, default)
    if val is None:
        return default
    return val

# -----------------------------
# Configuration (edit or set via .env)
# -----------------------------
BASE_URL = env("ARTHA_ACTIONS_BASEURL", "http://localhost:8787")
API_KEY  = env("ARTHA_ACTIONS_APIKEY", "")  # leave empty if not required
BUYER    = env("ARTHA_BUYER_PUBKEY", "BuyerWalletPublicKeyHere")
SELLER   = env("ARTHA_SELLER_PUBKEY", "SellerWalletPublicKeyHere")
MINT     = env("ARTHA_MINT", "USDC")
AMOUNT   = float(env("ARTHA_AMOUNT", "1.0"))
DELIVERY_BY_ISO = env("ARTHA_DELIVERY_BY", "2025-11-30T23:59:59Z")
DISPUTE_WINDOW_HOURS = int(env("ARTHA_DISPUTE_HOURS", "48"))

def headers(extra: dict | None = None) -> dict:
    h = {"content-type": "application/json"}
    if API_KEY:
        h["authorization"] = f"Bearer {API_KEY}"
    if extra:
        h.update(extra)
    return h

def show(title: str, obj: t.Any):
    print(f"\n### {title}")
    try:
        print(json.dumps(obj, indent=2)[:2000])
    except Exception:
        print(obj)

def request_json(method: str, path: str, body: dict | None = None) -> dict:
    url = f"{BASE_URL.rstrip('/')}/{path.lstrip('/')}"
    resp = requests.request(method, url, headers=headers(), json=body, timeout=30)
    if not resp.ok:
        raise RuntimeError(f"HTTP {resp.status_code} {resp.reason}: {resp.text[:500]}")
    return t.cast(dict, resp.json())

## 1) Health check

In [ ]:
health = request_json("GET", "/health")
show("Health", health)

## 2) Initiate a deal
Server should return a `dealId`. Some servers also return a `blinkUrl` or a serialized funding transaction.

In [ ]:
init_body = {
    "title": "Notebook escrow: demo gadget",
    "buyer": BUYER,
    "seller": SELLER,
    "amount": AMOUNT,
    "mint": MINT,
    "deliveryBy": DELIVERY_BY_ISO,
    "disputeWindowHours": DISPUTE_WINDOW_HOURS
}
init_res = request_json("POST", "/actions/deals/initiate", init_body)
show("Initiate", init_res)
deal_id = init_res.get("dealId", "")
assert deal_id, "Server did not return dealId"
blink_url = init_res.get("blinkUrl")
fund_tx  = init_res.get("tx")  # may be None
deal_id

## 3) Build funding transaction (if you didn’t use Blink)
If you already funded via a Blink from the initiate response, you can skip this cell.

In [ ]:
if not blink_url and not fund_tx:
    fund_res = request_json("POST", "/actions/deals/fund", {"dealId": deal_id, "payer": BUYER})
    show("Fund (tx or Blink)", fund_res)
else:
    show("Fund (skipped)", {"blinkUrl": blink_url, "hasTx": bool(fund_tx)})

## 4) Mark delivered
Seller provides minimal evidence. Replace the CID with a real one if you have it.

In [ ]:
deliver_res = request_json("POST", "/actions/deals/deliver", {
    "dealId": deal_id,
    "evidenceCid": "bafyREPLACE",
    "note": "Shipped via devnet courier"
})
show("Deliver", deliver_res)

## 5) Release funds to seller
Buyer approves release. If you’re testing a refund path, call `/actions/deals/refund` instead with `approver` set appropriately.

In [ ]:
release_res = request_json("POST", "/actions/deals/release", {
    "dealId": deal_id,
    "approver": BUYER
})
show("Release", release_res)

## 6) Fetch deal status

In [ ]:
status = request_json("GET", f"/actions/deals/{deal_id}")
show("Status", status)

### Appendix: Helper to sleep between steps if your server processes async
Not all servers block on-chain confirmation; add a tiny delay if needed.

In [ ]:
def pause(seconds: float = 2.0):
    print(f"Sleeping {seconds:.1f}s...")
    time.sleep(seconds)

pause(1.0)